# Historical Data Modeling

Notebook dedicado ao tuning de lags e hiperpar?metros para os targets multiclasses `target_{n}d`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.historical_loader import load_historical_asset_dataframe
from src.data.historical_preprocessing import prepare_historical_market_dataframe
from src.features.historical_features import DEFAULT_HISTORICAL_FEATURES, build_log_diff_dataset, create_multi_period_targets
from src.models.historical_training import (
    build_default_multiclass_models,
    build_default_param_spaces,
    find_best_lags,
    prepare_lagged_modeling_data,
    tune_models_with_best_lags,
    tune_models_with_cv,
)


In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'CSvs'
ASSET_NAME = 'Binance_BTCUSDT_d'
PERIODS = (1, 3, 7, 14, 30)
THRESHOLD_MULTIPLIER = 0.75
MAX_LAGS = 20
TRANSFORMED_FEATURE_COLUMNS = [
    'log_diff_open',
    'log_diff_high',
    'log_diff_low',
    'log_diff_close',
    'log_diff_volume_btc',
    'log_diff_volume_usdt',
    'log_diff_tradecount',
]
MODELS = build_default_multiclass_models()
MODEL_SPACES = build_default_param_spaces()


In [ ]:
raw_btc_hist = load_historical_asset_dataframe(DATA_DIR, ASSET_NAME)
btc_hist = prepare_historical_market_dataframe(raw_btc_hist)
btc_hist_transformed = build_log_diff_dataset(btc_hist, feature_columns=DEFAULT_HISTORICAL_FEATURES, keep_close_column=True)
df_with_targets = create_multi_period_targets(btc_hist_transformed, periods=PERIODS, threshold_multiplier=THRESHOLD_MULTIPLIER)


In [ ]:
results_two_step = {}
for horizon in PERIODS:
    target_column = f'target_{horizon}d'
    modeling_data = prepare_lagged_modeling_data(df_with_targets, TRANSFORMED_FEATURE_COLUMNS, MAX_LAGS, target_column)
    best_lags = find_best_lags(modeling_data, TRANSFORMED_FEATURE_COLUMNS, target_column, MAX_LAGS, MODELS)
    results_two_step[horizon] = tune_models_with_best_lags(
        modeling_data,
        TRANSFORMED_FEATURE_COLUMNS,
        target_column,
        best_lags,
        MODEL_SPACES,
    )
results_two_step


In [ ]:
results_joint_cv = {}
for horizon in PERIODS:
    target_column = f'target_{horizon}d'
    modeling_data = prepare_lagged_modeling_data(df_with_targets, TRANSFORMED_FEATURE_COLUMNS, MAX_LAGS, target_column)
    results_joint_cv[horizon] = tune_models_with_cv(
        modeling_data,
        TRANSFORMED_FEATURE_COLUMNS,
        target_column,
        MAX_LAGS,
        MODEL_SPACES,
    )
results_joint_cv


In [ ]:
import pickle
with open(PROJECT_ROOT / 'data' / 'historical_multiclass_results.pkl', 'wb') as fh:
    pickle.dump(results_joint_cv, fh)
